# LeetCode 174 — Dungeon Game (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def calculateMinimumHP(self, dungeon: List[List[int]]) -> int:

```

Đây là một bài **Hard** rất "đánh lừa": nhìn qua thấy giống bài tìm đường đi tổng nhỏ nhất trên lưới (LeetCode 64) — tưởng chỉ cần DP (quy hoạch động) từ góc trên-trái là xong. Nhưng nếu làm vậy thì **sai**. Notebook này sẽ cho bạn **tận mắt thấy nó sai ở đâu** (bằng phản ví dụ chạy được), rồi mới xây lời giải đúng: **DP đi ngược từ đích về**.

---

## Sau notebook này, bạn sẽ nắm được

1. Làm việc với **lưới 2 chiều** (list of lists): chỉ số `dungeon[i][j]`, đi phải / đi xuống, mô phỏng máu (HP) của hiệp sĩ trên một đường đi.
2. Công thức cho **một đường cố định**: `HP tối thiểu = max(1, 1 - (tổng tiền tố nhỏ nhất))`.
3. **Cách 1 — Brute force**: liệt kê mọi đường đi bằng DFS (đệ quy), và thấy số đường **bùng nổ** thế nào.
4. **Cách 2 — DP xuôi (SAI!)**: vì sao hai đại lượng "máu đang có" và "máu cần lúc đầu" **xung đột**, kèm 2 phản ví dụ cụ thể.
5. **Cách 3 — Đệ quy + memo** (ghi nhớ) từ ô `(i, j)` về đích.
6. **Cách 4 — DP bảng ngược** từ góc dưới-phải (cách chính): `need[i][j] = max(1, min(need dưới, need phải) - dungeon[i][j])`, trace đầy đủ trên ví dụ của đề.
7. **Cách 5 — DP một chiều**: chỉ giữ 1 hàng, bộ nhớ `O(n)`.
8. **Cách 6 — Binary search trên đáp án**: đoán HP ban đầu rồi kiểm tra khả thi bằng một DP xuôi (lần này DP xuôi lại **đúng** — và ta sẽ hiểu vì sao).
9. Lỗi hay gặp, mẹo nhận dạng pattern, bài liên quan và 6 bài tập có đáp án.

## 1. Đề bài (theo LeetCode)

Bọn quỷ đã bắt công chúa và nhốt nàng ở **góc dưới-phải** của một hầm ngục (dungeon). Hầm ngục gồm `m x n` phòng xếp thành lưới 2 chiều. Hiệp sĩ dũng cảm của chúng ta bắt đầu ở **phòng trên-trái** và phải chiến đấu xuyên qua hầm ngục để giải cứu công chúa.

Hiệp sĩ có một lượng **máu ban đầu** (initial health point) là một **số nguyên dương**. Nếu tại bất kỳ thời điểm nào máu của hiệp sĩ **giảm xuống 0 hoặc thấp hơn**, chàng **chết ngay lập tức**.

Một số phòng có quỷ canh giữ (biểu diễn bằng **số nguyên âm**), nên hiệp sĩ **mất máu** khi bước vào các phòng này; các phòng khác hoặc trống (biểu diễn bằng **0**), hoặc chứa quả cầu phép thuật giúp **tăng máu** (biểu diễn bằng **số nguyên dương**).

Để tới chỗ công chúa nhanh nhất có thể, hiệp sĩ quyết định mỗi bước **chỉ đi sang phải hoặc đi xuống**.

Hãy trả về **lượng máu ban đầu tối thiểu** để hiệp sĩ cứu được công chúa.

**Lưu ý:** phòng nào cũng có thể chứa quỷ hoặc quả cầu tăng máu, **kể cả phòng đầu tiên** hiệp sĩ bước vào và **phòng dưới-phải** nơi công chúa bị giam.

**Example 1**

```text
Input:  dungeon = [[-2,-3,3],[-5,-10,1],[10,30,-5]]
Output: 7
Explanation: The initial health of the knight must be at least 7 if he follows
the optimal path: RIGHT-> RIGHT -> DOWN -> DOWN.
```

**Example 2**

```text
Input:  dungeon = [[0]]
Output: 1
```

**Constraints**

- `m == dungeon.length`
- `n == dungeon[i].length`
- `1 <= m, n <= 200`
- `-1000 <= dungeon[i][j] <= 1000`

Ba điều cần để ý ngay từ đầu:

- Máu **bằng 0 cũng là chết**. Vậy "sống" nghĩa là máu `>= 1` ở **mọi** thời điểm.
- Máu ban đầu phải là **số nguyên dương**, nên đáp án **luôn `>= 1`** (Example 2: phòng trống, vẫn phải trả về 1 chứ không phải 0).
- Hiệp sĩ chịu tác động của **cả phòng đầu** `(0, 0)` lẫn **phòng cuối** `(m-1, n-1)`.

In [1]:
from typing import List   # LeetCode tự import sẵn; chạy ngoài LeetCode thì ta tự import

dungeon1 = [[-2, -3, 3],
            [-5, -10, 1],
            [10, 30, -5]]
dungeon2 = [[0]]

print("Example 1:", dungeon1, "-> mong đợi 7")
print("Example 2:", dungeon2, "-> mong đợi 1")
print("Số hàng m =", len(dungeon1), "| số cột n =", len(dungeon1[0]))

Example 1: [[-2, -3, 3], [-5, -10, 1], [10, 30, -5]] -> mong đợi 7
Example 2: [[0]] -> mong đợi 1
Số hàng m = 3 | số cột n = 3


## 2. Kiến thức nền 1: lưới 2 chiều trong Python (list of lists)

Lưới `m x n` được lưu bằng một **list chứa `m` list con**, mỗi list con là **một hàng** có `n` phần tử.

- `dungeon[i]` là **hàng thứ `i`** (đếm từ 0, từ trên xuống).
- `dungeon[i][j]` là ô ở **hàng `i`, cột `j`** (cột đếm từ 0, từ trái sang).
- `m = len(dungeon)` (số hàng), `n = len(dungeon[0])` (số cột của hàng đầu — mọi hàng dài bằng nhau).

Từ ô `(i, j)`, hiệp sĩ chỉ có hai nước đi:

| nước đi | ký hiệu | ô mới | hợp lệ khi |
|---|---|---|---|
| sang phải (RIGHT) | `R` | `(i, j + 1)` | `j + 1 < n` |
| đi xuống (DOWN) | `D` | `(i + 1, j)` | `i + 1 < m` |

Đích là ô `(m - 1, n - 1)`. Mọi đường đi đều gồm đúng `n - 1` bước `R` và `m - 1` bước `D` (thứ tự tuỳ ý), nên **đi qua đúng `m + n - 1` phòng** (tính cả phòng đầu và phòng cuối).

In [2]:
grid = [[-2, -3, 3],
        [-5, -10, 1],
        [10, 30, -5]]

print("grid[0]     =", grid[0])        # hàng đầu tiên
print("grid[1][2]  =", grid[1][2])     # hàng 1, cột 2 -> 1
print("grid[2][1]  =", grid[2][1])     # hàng 2, cột 1 -> 30 (quả cầu to!)
print("m, n        =", len(grid), len(grid[0]))
print("ô đích      =", grid[len(grid) - 1][len(grid[0]) - 1])   # -5: công chúa cũng bị quỷ canh!

grid[0]     = [-2, -3, 3]
grid[1][2]  = 1
grid[2][1]  = 30
m, n        = 3 3
ô đích      = -5


Để dễ nhìn, ta viết một hàm **in lưới** và (tuỳ chọn) đánh dấu một đường đi bằng dấu ngoặc vuông `[ ]`. Đường đi được cho dưới dạng chuỗi nước đi, ví dụ `"RRDD"` nghĩa là phải, phải, xuống, xuống.

Hàm `path_cells` đổi chuỗi nước đi thành danh sách toạ độ các phòng đi qua — bạn sẽ dùng nó nhiều lần trong notebook.

In [3]:
def path_cells(moves):
    """Đổi chuỗi nước đi ('R'/'D') thành list toạ độ các phòng đi qua, bắt đầu từ (0, 0)."""
    i, j = 0, 0
    cells = [(0, 0)]
    for mv in moves:
        if mv == "R":
            j += 1
        else:          # "D"
            i += 1
        cells.append((i, j))
    return cells


def show_grid(dungeon, moves=None):
    """In lưới; nếu có moves thì các phòng trên đường đi được bọc trong [ ]."""
    on_path = set(path_cells(moves)) if moves is not None else set()
    for i, row in enumerate(dungeon):
        parts = []
        for j, v in enumerate(row):
            parts.append(f"[{v:>3}]" if (i, j) in on_path else f" {v:>3} ")
        print(" ".join(parts))


print(path_cells("RRDD"))
show_grid(dungeon1, "RRDD")

[(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)]
[ -2] [ -3] [  3]
  -5   -10  [  1]
  10    30  [ -5]


## 3. Kiến thức nền 2: mô phỏng hiệp sĩ trên **một đường cố định**

Trước khi tìm đường tốt nhất, hãy làm việc nhỏ hơn: **cho trước một đường đi và một lượng máu ban đầu `hp`**, hiệp sĩ có sống sót không?

Quy tắc: mỗi khi bước vào một phòng, `hp += giá trị phòng`; nếu `hp <= 0` thì chết ngay.

Thử tay với Example 1, đường `RRDD`, giá trị các phòng lần lượt là `-2, -3, 3, 1, -5`:

| phòng | giá trị | hp (bắt đầu 7) | hp (bắt đầu 6) |
|---|---:|---:|---:|
| (0,0) | -2 | 5 | 4 |
| (0,1) | -3 | 2 | 1 |
| (0,2) | +3 | 5 | 4 |
| (1,2) | +1 | 6 | 5 |
| (2,2) | -5 | **1** → sống | **0** → chết! |

Vậy với đường này, 7 máu là đủ, 6 máu là không đủ. Hàm dưới in đúng bảng đó.

In [4]:
def simulate_walk(dungeon, moves, hp, verbose=True):
    """Cho hiệp sĩ đi theo moves với máu ban đầu hp. Trả về True nếu sống tới đích."""
    for (i, j) in path_cells(moves):
        hp += dungeon[i][j]
        if verbose:
            print(f"  vào ({i},{j}) giá trị {dungeon[i][j]:>4} -> hp = {hp}")
        if hp <= 0:
            if verbose:
                print("  => CHẾT")
            return False
    if verbose:
        print("  => SỐNG, cứu được công chúa")
    return True


print("Bắt đầu với 7 máu:")
simulate_walk(dungeon1, "RRDD", 7)
print("Bắt đầu với 6 máu:")
_ = simulate_walk(dungeon1, "RRDD", 6)   # gán vào _ để Jupyter không in thêm True/False

Bắt đầu với 7 máu:
  vào (0,0) giá trị   -2 -> hp = 5
  vào (0,1) giá trị   -3 -> hp = 2
  vào (0,2) giá trị    3 -> hp = 5
  vào (1,2) giá trị    1 -> hp = 6
  vào (2,2) giá trị   -5 -> hp = 1
  => SỐNG, cứu được công chúa
Bắt đầu với 6 máu:
  vào (0,0) giá trị   -2 -> hp = 4
  vào (0,1) giá trị   -3 -> hp = 1
  vào (0,2) giá trị    3 -> hp = 4
  vào (1,2) giá trị    1 -> hp = 5
  vào (2,2) giá trị   -5 -> hp = 0
  => CHẾT


### Từ mô phỏng tới công thức: tổng tiền tố (prefix sum)

Gọi các giá trị trên đường là `v1, v2, ..., vk`. Sau khi vào phòng thứ `t`, máu là:

```text
hp_t = H + (v1 + v2 + ... + vt) = H + prefix[t]
```

trong đó `H` là máu ban đầu và `prefix[t]` là **tổng tiền tố** (prefix sum) — tổng của `t` phần tử đầu.

Điều kiện sống: `hp_t >= 1` với **mọi** `t`, tức là `H >= 1 - prefix[t]` với mọi `t`. Chỉ cần thoả cho prefix **nhỏ nhất** (chỗ "đáy" thấp nhất của đường) là thoả cho tất cả. Thêm điều kiện `H >= 1` (máu ban đầu dương), ta được:

```text
H_min(đường) = max(1, 1 - min(prefix))
```

**Ví dụ số** — đường `RRDD`: prefix = `[-2, -5, -2, -1, -6]`, nhỏ nhất là `-6` → `H_min = max(1, 1 - (-6)) = 7`. Khớp mô phỏng!

**Ví dụ có `max(1, ...)` phát huy tác dụng** — đường toàn quả cầu `[5, 2]`: prefix = `[5, 7]`, `1 - 5 = -4` → `H_min = max(1, -4) = 1`. Máu ban đầu không được là 0 hay âm.

Python có sẵn `itertools.accumulate` để tính prefix sum:

In [5]:
from itertools import accumulate

values = [-2, -3, 3, 1, -5]
prefix = list(accumulate(values))     # cộng dồn: -2, -2-3, -2-3+3, ...
print("values =", values)
print("prefix =", prefix)
print("min(prefix) =", min(prefix))


def min_hp_for_values(values):
    """HP ban đầu nhỏ nhất để đi qua dãy phòng values mà không chết."""
    return max(1, 1 - min(accumulate(values)))


print("H_min cho RRDD   :", min_hp_for_values(values))   # 7
print("H_min cho [5, 2] :", min_hp_for_values([5, 2]))   # 1
print("H_min cho [0]    :", min_hp_for_values([0]))      # 1 (Example 2)

values = [-2, -3, 3, 1, -5]
prefix = [-2, -5, -2, -1, -6]
min(prefix) = -6
H_min cho RRDD   : 7
H_min cho [5, 2] : 1
H_min cho [0]    : 1


Bài toán 1 chiều này chính là LeetCode 1413 — *Minimum Value to Get Positive Step by Step Sum*. Bài 174 khó hơn ở chỗ: ta được **chọn đường**, và số đường đi thì rất nhiều. Bài toán thật sự là:

```text
đáp án = min over mọi đường P của  max(1, 1 - min(prefix của P))
```

Câu hỏi tiếp theo: **liệt kê mọi đường** như thế nào?

## 4. Kiến thức nền 3: đệ quy và DFS để liệt kê đường đi

**Đệ quy** (recursion) là khi một hàm **gọi lại chính nó** với bài toán nhỏ hơn, cho tới khi gặp **trường hợp cơ sở** (base case) thì dừng.

**DFS** (Depth-First Search — tìm kiếm theo chiều sâu) trên lưới: từ ô hiện tại, thử đi **hết một hướng** cho tới đích, rồi **quay lui** (backtrack) thử hướng còn lại.

Liệt kê mọi đường từ `(i, j)` tới đích:

- **Base case:** đang ở đích `(m-1, n-1)` → ghi lại đường đã đi.
- **Bước đệ quy:** nếu đi phải được thì thêm `"R"` và gọi đệ quy ở `(i, j+1)`; nếu đi xuống được thì thêm `"D"` và gọi ở `(i+1, j)`.

Thử với lưới nhỏ `2 x 3`: cần 2 bước `R`, 1 bước `D` → phải có 3 đường: `RRD`, `RDR`, `DRR`.

In [6]:
def list_paths(m, n):
    """Trả về list mọi chuỗi nước đi từ (0,0) tới (m-1,n-1) trên lưới m x n."""
    result = []

    def dfs(i, j, moves):
        if i == m - 1 and j == n - 1:      # base case: tới đích
            result.append(moves)
            return
        if j + 1 < n:
            dfs(i, j + 1, moves + "R")     # thử đi phải trước
        if i + 1 < m:
            dfs(i + 1, j, moves + "D")     # rồi thử đi xuống

    dfs(0, 0, "")
    return result


print("2 x 3:", list_paths(2, 3))
print("3 x 3:", list_paths(3, 3))

2 x 3: ['RRD', 'RDR', 'DRR']
3 x 3: ['RRDD', 'RDRD', 'RDDR', 'DRRD', 'DRDR', 'DDRR']


Có bao nhiêu đường trên lưới `m x n`? Mỗi đường là một cách sắp xếp `m-1` chữ `D` và `n-1` chữ `R` → số đường là **tổ hợp** `C(m+n-2, m-1)`. Python có sẵn `math.comb(a, b)` = "chọn `b` phần tử từ `a` phần tử".

In [7]:
import math

for m, n in [(2, 3), (3, 3), (5, 5), (10, 10), (15, 15), (200, 200)]:
    total = math.comb(m + n - 2, m - 1)
    assert m * n > 100 or total == len(list_paths(m, n))   # kiểm lại công thức với lưới nhỏ
    shown = total if total < 10**12 else f"khoảng 10^{len(str(total)) - 1}"
    print(f"{m:>3} x {n:<3} -> số đường = {shown}")

  2 x 3   -> số đường = 3
  3 x 3   -> số đường = 6
  5 x 5   -> số đường = 70
 10 x 10  -> số đường = 48620
 15 x 15  -> số đường = 40116600
200 x 200 -> số đường = khoảng 10^118


## 5. Cách 1 — Brute force: thử mọi đường đi

Ý tưởng tự nhiên nhất: **liệt kê mọi đường**, với mỗi đường tính `H_min(đường)` bằng công thức prefix ở mục 3, rồi lấy **nhỏ nhất**.

Thay vì tạo cả chuỗi đường đi rồi mới tính, ta cho DFS **mang theo hai con số** khi đi:

- `s` — tổng các phòng đã đi qua (chính là prefix hiện tại),
- `low` — prefix nhỏ nhất gặp được tới giờ (đáy thấp nhất).

Tới đích thì `H_min = max(1, 1 - low)`, cập nhật đáp án tốt nhất.

In [8]:
def calculate_bruteforce(dungeon: List[List[int]]) -> int:
    m, n = len(dungeon), len(dungeon[0])
    best = float("inf")

    def dfs(i, j, s, low):
        nonlocal best                      # cho phép sửa biến best của hàm ngoài
        s += dungeon[i][j]                 # bước vào phòng (i, j)
        low = min(low, s)                  # cập nhật đáy thấp nhất
        if i == m - 1 and j == n - 1:
            best = min(best, max(1, 1 - low))
            return
        if j + 1 < n:
            dfs(i, j + 1, s, low)
        if i + 1 < m:
            dfs(i + 1, j, s, low)

    dfs(0, 0, 0, float("inf"))
    return best


print(calculate_bruteforce(dungeon1))   # 7
print(calculate_bruteforce(dungeon2))   # 1

7
1


Hai chi tiết Python mới:

- `float("inf")` là **vô cực dương**: lớn hơn mọi số. Dùng làm giá trị khởi tạo cho "nhỏ nhất" (`min(inf, 5) = 5`).
- `nonlocal best`: hàm con `dfs` muốn **gán lại** biến `best` của hàm cha. Không có dòng này, Python sẽ coi `best` trong `dfs` là biến mới cục bộ và báo lỗi.

### Trace cách 1 trên Example 1

Lưới 3 x 3 có 6 đường. Bảng dưới (tính tay) cho thấy mỗi đường cần bao nhiêu máu:

| đường | giá trị các phòng | prefix | min prefix | H_min |
|---|---|---|---:|---:|
| RRDD | -2, -3, 3, 1, -5 | -2, -5, -2, -1, -6 | -6 | **7** |
| RDRD | -2, -3, -10, 1, -5 | -2, -5, -15, -14, -19 | -19 | 20 |
| RDDR | -2, -3, -10, 30, -5 | -2, -5, -15, 15, 10 | -15 | 16 |
| DRRD | -2, -5, -10, 1, -5 | -2, -7, -17, -16, -21 | -21 | 22 |
| DRDR | -2, -5, -10, 30, -5 | -2, -7, -17, 13, 8 | -17 | 18 |
| DDRR | -2, -5, 10, 30, -5 | -2, -7, 3, 33, 28 | -7 | 8 |

Đáp án = `min(7, 20, 16, 22, 18, 8) = 7`, đạt ở đường `RRDD` — đúng như đề nói. **Để ý đường `DDRR`**: nó kết thúc với **nhiều máu dư nhất** (prefix cuối = 28) nhưng lại cần 8 máu vì "đáy" -7 sâu hơn. Ghi nhớ chi tiết này, ta sẽ cần nó ở mục 6.

In [9]:
def trace_bruteforce(dungeon):
    m, n = len(dungeon), len(dungeon[0])
    best = None
    print(f"{'đường':<6} | {'prefix':<26} | low | H_min")
    for moves in list_paths(m, n):
        values = [dungeon[i][j] for (i, j) in path_cells(moves)]
        prefix = list(accumulate(values))
        h = max(1, 1 - min(prefix))
        print(f"{moves:<6} | {str(prefix):<26} | {min(prefix):>3} | {h:>4}")
        if best is None or h < best[0]:
            best = (h, moves)
    print("=> tốt nhất:", best)
    return best[0]


_ = trace_bruteforce(dungeon1)   # gán vào _ để Jupyter không in thêm giá trị return

đường  | prefix                     | low | H_min
RRDD   | [-2, -5, -2, -1, -6]       |  -6 |    7
RDRD   | [-2, -5, -15, -14, -19]    | -19 |   20
RDDR   | [-2, -5, -15, 15, 10]      | -15 |   16
DRRD   | [-2, -7, -17, -16, -21]    | -21 |   22
DRDR   | [-2, -7, -17, 13, 8]       | -17 |   18
DDRR   | [-2, -7, 3, 33, 28]        |  -7 |    8
=> tốt nhất: (7, 'RRDD')


### Cách 1 vướng ở đâu?

- **Time:** `O(C(m+n-2, m-1) × (m+n))` — số đường nhân độ dài mỗi đường. Với `m = n = 200` số đường khoảng `10^118`: vũ trụ kết thúc trước khi chạy xong.
- **Space:** `O(m + n)` cho độ sâu đệ quy.

Cell dưới đo thời gian trên lưới vuông nhỏ dần lớn lên — để ý mỗi khi cạnh tăng 1, thời gian tăng gần **4 lần**.

**Nút thắt:** rất nhiều đường **dùng chung đoạn cuối**. Ví dụ `RDRD` và `DRRD` đều đi qua `(1,1) → (1,2) → (2,2)`, nhưng brute force tính lại đoạn chung đó cho từng đường. Muốn tái sử dụng, ta cần một đại lượng gắn với **ô**, không phải với **đường**.

In [10]:
import random
import time

random.seed(0)
for size in range(6, 12):
    g = [[random.randint(-10, 10) for _ in range(size)] for _ in range(size)]
    t0 = time.perf_counter()
    calculate_bruteforce(g)
    dt = time.perf_counter() - t0
    print(f"{size:>2} x {size:<2}: {math.comb(2 * size - 2, size - 1):>7} đường, {dt * 1000:8.1f} ms")

 6 x 6 :     252 đường,      0.2 ms
 7 x 7 :     924 đường,      0.5 ms
 8 x 8 :    3432 đường,      1.9 ms
 9 x 9 :   12870 đường,      8.7 ms
10 x 10:   48620 đường,     25.9 ms


11 x 11:  184756 đường,    105.7 ms


## 6. Cách 2 — Thử DP xuôi từ trên-trái (và vì sao nó SAI)

Nếu bạn đã học LeetCode 64 (Minimum Path Sum), phản xạ tự nhiên là: **đi từ `(0,0)` xuống `(m-1,n-1)`**, tại mỗi ô chỉ giữ lại **"đường tốt nhất" đi tới ô đó** (chọn giữa đường đến từ trên và đường đến từ trái).

Nhưng "tốt nhất" theo tiêu chí nào? Một đường đi tới ô `(i, j)` có **hai con số** quan trọng:

- `s` — **máu đang có thêm** (tổng tới giờ, càng lớn càng tốt cho tương lai),
- `low` — **đáy thấp nhất** đã gặp, quyết định máu cần lúc đầu `max(1, 1 - low)` (càng cao càng tốt cho quá khứ).

Ta thử cả hai cách chọn "tốt nhất":

- **Phiên bản A** — giữ đường có `s` **lớn nhất** (nhiều máu dư nhất).
- **Phiên bản B** — giữ đường có `low` **lớn nhất** (cần ít máu ban đầu nhất).

Cả hai đều có code ngắn gọn y như Minimum Path Sum. Mỗi ô lưu một cặp `(s, low)`.

In [11]:
def forward_dp(dungeon, key):
    """DP xuôi: tại mỗi ô chỉ giữ 1 cặp (s, low) 'tốt nhất' theo key. CỐ Ý SAI để minh hoạ."""
    m, n = len(dungeon), len(dungeon[0])
    state = [[None] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            if i == 0 and j == 0:
                candidates = [(0, float("inf"))]          # chưa vào phòng nào
            else:
                candidates = []
                if i > 0:
                    candidates.append(state[i - 1][j])    # đến từ trên
                if j > 0:
                    candidates.append(state[i][j - 1])    # đến từ trái
            s, low = max(candidates, key=key)             # chọn 1 cặp "tốt nhất"
            s += dungeon[i][j]
            state[i][j] = (s, min(low, s))
    s, low = state[m - 1][n - 1]
    return max(1, 1 - low)


def forward_dp_max_sum(dungeon):      # Phiên bản A: ưu tiên s lớn
    return forward_dp(dungeon, key=lambda t: (t[0], t[1]))


def forward_dp_min_need(dungeon):     # Phiên bản B: ưu tiên low lớn (cần ít máu)
    return forward_dp(dungeon, key=lambda t: (t[1], t[0]))


print("Example 1: đúng =", calculate_bruteforce(dungeon1),
      "| A (max s) =", forward_dp_max_sum(dungeon1),
      "| B (min need) =", forward_dp_min_need(dungeon1))

Example 1: đúng = 7 | A (max s) = 8 | B (min need) = 7


`key=lambda t: (t[0], t[1])` nghĩa là: so sánh các cặp theo `t[0]` trước, hoà thì so `t[1]`. `lambda` là cách viết **hàm vô danh** một dòng: `lambda t: t[0]` tương đương `def f(t): return t[0]`.

### Phản ví dụ 1 — Phiên bản A sai ngay trên Example 1 của đề

Phiên bản A trả về **8** thay vì 7. Vì sao? Nó luôn giữ đường có nhiều máu dư nhất, nên cuối cùng chọn đúng đường `DDRR` (prefix cuối = 28, lớn nhất trong 6 đường ở bảng mục 5). Nhưng `DDRR` có đáy -7, cần 8 máu. Đường đúng `RRDD` chỉ dư 1 máu lúc cuối nhưng đáy chỉ -6.

Nói cách khác: **"nhiều máu lúc cuối" không có nghĩa là "cần ít máu lúc đầu"**.

In [12]:
print("Phiên bản A chọn đường kiểu này (DDRR):")
show_grid(dungeon1, "DDRR")
print("H_min(DDRR) =", min_hp_for_values([dungeon1[i][j] for i, j in path_cells("DDRR")]))
print()
print("Đường đúng (RRDD):")
show_grid(dungeon1, "RRDD")
print("H_min(RRDD) =", min_hp_for_values([dungeon1[i][j] for i, j in path_cells("RRDD")]))

Phiên bản A chọn đường kiểu này (DDRR):
[ -2]   -3     3 
[ -5]  -10     1 
[ 10] [ 30] [ -5]
H_min(DDRR) = 8

Đường đúng (RRDD):
[ -2] [ -3] [  3]
  -5   -10  [  1]
  10    30  [ -5]
H_min(RRDD) = 7


### Phản ví dụ 2 — Phiên bản B cũng sai

Phiên bản B đúng trên Example 1 (do may mắn), nhưng sai trên lưới `2 x 3` nhỏ xíu này:

```text
dungeon = [[0, -1,  3],
           [0,  0, -3]]
```

| đường | giá trị | prefix | low | H_min |
|---|---|---|---:|---:|
| RRD | 0, -1, 3, -3 | 0, -1, 2, -1 | -1 | **2** |
| RDR | 0, -1, 0, -3 | 0, -1, -1, -4 | -4 | 5 |
| DRR | 0, 0, 0, -3 | 0, 0, 0, -3 | -3 | 4 |

Đáp án đúng là **2** (đường `RRD`). Phiên bản B làm gì?

1. Tại ô `(1,1)`: đến từ trên `(0,1)` có `(s=-1, low=-1)`, đến từ trái `(1,0)` có `(s=0, low=0)`. B chọn từ trái vì `low = 0` cao hơn. Ổn.
2. Tại ô đích `(1,2)`: đến từ trên `(0,2)` có `(s=2, low=-1)` — **cần 2 máu nhưng dư tới 2**; đến từ trái `(1,1)` có `(s=0, low=0)` — **cần 1 máu nhưng không dư gì**. B tham lam chọn cái cần 1 máu.
3. Bước vào phòng đích -3: `s = 0 - 3 = -3` → cần **4** máu. Trong khi nếu giữ đường từ trên: `s = 2 - 3 = -1` → chỉ cần **2**.

Đường "rẻ" lúc trước không còn máu dự trữ để chịu cú đánh cuối.

In [13]:
counter2 = [[0, -1, 3],
            [0, 0, -3]]

_ = trace_bruteforce(counter2)
print("Phiên bản A (max s)   :", forward_dp_max_sum(counter2))
print("Phiên bản B (min need):", forward_dp_min_need(counter2))

đường  | prefix                     | low | H_min
RRD    | [0, -1, 2, -1]             |  -1 |    2
RDR    | [0, -1, -1, -4]            |  -4 |    5
DRR    | [0, 0, 0, -3]              |  -3 |    4
=> tốt nhất: (2, 'RRD')
Phiên bản A (max s)   : 2
Phiên bản B (min need): 4


### Gốc rễ vấn đề: hai đại lượng xung đột, không có "tốt nhất" duy nhất

Khi đi xuôi, một đường tới ô `(i, j)` được mô tả bởi **cặp** `(s, low)`:

- Muốn **quá khứ rẻ** → cần `low` lớn.
- Muốn **tương lai an toàn** → cần `s` lớn (để chịu được quỷ phía trước).

Để ý: trên lưới này phiên bản A lại đúng, còn trên Example 1 thì B đúng — mỗi phiên bản thua ở một kiểu lưới khác nhau, nên không thể "vá" bằng cách chọn khéo tiêu chí.

Hai đường `(s=2, low=-1)` và `(s=0, low=0)` **không so được**: mỗi đường hơn ở một mặt. Đường nào tốt hơn còn **tuỳ vào phần đường phía sau** — mà khi đi xuôi ta chưa biết phía sau! Nên giữ 1 cặp mỗi ô là **mất thông tin**, còn giữ tất cả các cặp thì lại quay về brute force.

**Đảo hướng nhìn.** Thay vì hỏi "đi từ đầu tới `(i, j)` thì sao?", hãy hỏi:

> **"Đứng trước cửa phòng `(i, j)`, cần tối thiểu bao nhiêu máu để đi nốt tới công chúa?"**

Câu trả lời chỉ phụ thuộc vào **phần đường phía sau** (từ `(i, j)` tới đích), hoàn toàn **không phụ thuộc** ta đã tới `(i, j)` bằng cách nào. Đó là **một con số duy nhất** cho mỗi ô, và "càng nhỏ càng tốt" — không còn xung đột. Đây là cấu trúc con tối ưu (optimal substructure) mà DP cần.

## 7. Cách 3 — Đệ quy + memo từ ô `(i, j)` về đích

### Định nghĩa

`need(i, j)` = lượng máu **tối thiểu ngay trước khi bước vào** phòng `(i, j)` để đi tiếp tới công chúa mà không chết.

Đáp án của bài là `need(0, 0)`.

### Công thức truy hồi (lập luận từng bước)

Giả sử đứng trước phòng `(i, j)` với `h` máu, phòng có giá trị `d = dungeon[i][j]`.

1. Bước vào phòng: máu thành `h + d`.
2. Sau đó phải đi tiếp sang ô kế (phải hoặc dưới). Chọn ô kế **cần ít máu hơn**: `nxt = min(need(i+1, j), need(i, j+1))`.
3. Muốn sống: `h + d >= nxt` → `h >= nxt - d`.
4. Máu luôn phải `>= 1` (máu ban đầu dương, và đang sống) → `h >= 1`.

Gộp lại:

```text
need(i, j) = max(1, min(need(i+1, j), need(i, j+1)) - dungeon[i][j])
```

**Phòng công chúa** `(m-1, n-1)`: không còn đi đâu nữa, chỉ cần sau khi vào phòng còn `>= 1` máu → coi như `nxt = 1`: `need = max(1, 1 - d)`.

**Ô ngoài lưới** (đi phải khi đã ở cột cuối, hoặc xuống khi ở hàng cuối): không đi được → coi `need = +∞` để `min` không bao giờ chọn nó.

### Ví dụ số với công thức

- `d = -5`, ô kế cần 2: `max(1, 2 - (-5)) = 7`. Vào phòng mất 5, còn phải giữ 2 → cần 7. Hợp lý.
- `d = +30`, ô kế cần 1: `max(1, 1 - 30) = max(1, -29) = 1`. Quả cầu lớn **không thể "trả nợ trước"**: bạn vẫn phải còn sống để bước vào phòng này, nên tối thiểu là 1.
- `d = +3`, ô kế cần 5: `max(1, 5 - 3) = 2`. Quả cầu bù được 3 trong 5 máu cần.

In [14]:
def one_step(d, nxt):
    """Máu cần trước phòng có giá trị d, khi ô kế tiếp cần nxt máu."""
    return max(1, nxt - d)


print(one_step(-5, 2))    # 7
print(one_step(30, 1))    # 1  (không phải -29!)
print(one_step(3, 5))     # 2
print(one_step(-5, 1))    # 6  = need của phòng công chúa trong Example 1

7
1
2
6


### Kiến thức nền: memoization (ghi nhớ kết quả)

Viết thẳng công thức thành đệ quy thì **đúng** nhưng **chậm**: `need(1,1)` được gọi lại nhiều lần (từ `need(0,1)` và từ `need(1,0)`), mỗi lần lại tính lại cả cây con phía sau. Số lần gọi tăng theo hàm mũ — giống hệt brute force.

**Memoization** (ghi nhớ): lần đầu tính xong `need(i, j)` thì **lưu vào một dict** với key `(i, j)`; lần sau gặp lại chỉ việc tra. Mỗi ô chỉ tính đúng **1 lần** → tổng cộng `m × n` lần.

Cell dưới đếm số lần gọi hàm để thấy rõ khác biệt.

In [15]:
def count_calls(dungeon, use_memo):
    m, n = len(dungeon), len(dungeon[0])
    INF = float("inf")
    memo = {}
    calls = 0

    def need(i, j):
        nonlocal calls
        if i >= m or j >= n:
            return INF                                   # ô ngoài lưới
        if use_memo and (i, j) in memo:
            return memo[(i, j)]                          # đã tính rồi -> tra luôn
        calls += 1
        if i == m - 1 and j == n - 1:
            res = max(1, 1 - dungeon[i][j])              # phòng công chúa
        else:
            res = max(1, min(need(i + 1, j), need(i, j + 1)) - dungeon[i][j])
        memo[(i, j)] = res
        return res

    return need(0, 0), calls


random.seed(1)
for size in [3, 6, 9, 12]:
    g = [[random.randint(-10, 10) for _ in range(size)] for _ in range(size)]
    a, c1 = count_calls(g, use_memo=False)
    b, c2 = count_calls(g, use_memo=True)
    assert a == b
    print(f"{size:>2} x {size:<2}: không memo {c1:>8} lần tính | có memo {c2:>4} lần tính")

 3 x 3 : không memo       19 lần tính | có memo    9 lần tính
 6 x 6 : không memo      923 lần tính | có memo   36 lần tính
 9 x 9 : không memo    48619 lần tính | có memo   81 lần tính


12 x 12: không memo  2704155 lần tính | có memo  144 lần tính


Python có sẵn **decorator** `functools.lru_cache(maxsize=None)` làm đúng việc của `memo` dict: đặt `@lru_cache(maxsize=None)` ngay trên hàm, Python tự lưu kết quả theo tham số. (Decorator là cú pháp `@ten` đặt trên `def`, "bọc" hàm bằng thêm một hành vi.)

Mẹo nhỏ: định nghĩa hàm đệ quy **bên trong** hàm chính, để mỗi lần gọi với lưới mới thì có một cache mới (không dùng nhầm kết quả của lưới cũ).

**Độ sâu đệ quy:** mỗi lần gọi tiến 1 bước về đích, nên sâu tối đa `m + n - 1 <= 399` tầng — dưới giới hạn mặc định 1000 của Python, an toàn với constraints.

In [16]:
from functools import lru_cache


def calculate_memo(dungeon: List[List[int]]) -> int:
    m, n = len(dungeon), len(dungeon[0])
    INF = float("inf")

    @lru_cache(maxsize=None)
    def need(i, j):
        if i >= m or j >= n:
            return INF
        if i == m - 1 and j == n - 1:
            return max(1, 1 - dungeon[i][j])
        return max(1, min(need(i + 1, j), need(i, j + 1)) - dungeon[i][j])

    return need(0, 0)


print(calculate_memo(dungeon1))   # 7
print(calculate_memo(dungeon2))   # 1
print(calculate_memo(counter2))   # 2

7
1
2


**Độ phức tạp cách 3** (với `m` hàng, `n` cột):

- **Time:** `O(m × n)` — mỗi ô tính 1 lần, mỗi lần `O(1)`.
- **Space:** `O(m × n)` cho cache + `O(m + n)` cho call stack (ngăn xếp lời gọi đệ quy).

Cách này đã đủ nhanh để nộp. Nhưng đệ quy có chi phí gọi hàm và rủi ro độ sâu; bước tiếp theo là **lật ngược** nó thành vòng lặp điền bảng.

## 8. Cách 4 — DP bảng ngược từ góc dưới-phải (cách chính)

Đệ quy + memo tính `need(i, j)` sau khi đã có `need(i+1, j)` (ô dưới) và `need(i, j+1)` (ô phải). Vậy nếu ta **tự điền bảng** theo thứ tự "dưới và phải xong trước", ta khỏi cần đệ quy:

- duyệt hàng `i` từ `m-1` **lùi về** `0`,
- trong mỗi hàng, duyệt cột `j` từ `n-1` **lùi về** `0`.

### Kiến thức nền: tạo bảng 2 chiều đúng cách

Có một cái bẫy kinh điển khi tạo bảng 2 chiều trong Python: `[[0] * n] * m` tạo ra `m` tham chiếu tới **cùng một list hàng**! Sửa một ô là cả cột đổi theo. Cách đúng là list comprehension `[[0] * n for _ in range(m)]` — mỗi vòng tạo một list hàng mới.

In [17]:
bad = [[0] * 3] * 2                    # SAI: 2 hàng là cùng một object
good = [[0] * 3 for _ in range(2)]     # ĐÚNG: 2 hàng độc lập

bad[0][0] = 99
good[0][0] = 99
print("bad  =", bad, "| bad[0] is bad[1]  :", bad[0] is bad[1])
print("good =", good, "| good[0] is good[1]:", good[0] is good[1])

print("Duyệt lùi:", list(range(3 - 1, -1, -1)))   # range(start, stop, step): 2, 1, 0

bad  = [[99, 0, 0], [99, 0, 0]] | bad[0] is bad[1]  : True
good = [[99, 0, 0], [0, 0, 0]] | good[0] is good[1]: False
Duyệt lùi: [2, 1, 0]


### Mẹo "hàng rào" (sentinel): thêm 1 hàng và 1 cột

Để khỏi viết nhiều `if` cho hàng cuối / cột cuối / phòng công chúa, ta tạo bảng **kích thước `(m+1) x (n+1)`**:

- mọi ô mặc định là `INF` (hàng `m` và cột `n` là "ngoài lưới" — không đi được),
- **trừ hai ô kề phòng công chúa**: `need[m][n-1] = 1` (ngay dưới) và `need[m-1][n] = 1` (ngay phải).

Nhờ vậy phòng công chúa cũng dùng **chung công thức**: `min(need[m][n-1], need[m-1][n]) = min(1, 1) = 1` → `need = max(1, 1 - d)`, đúng như base case ở mục 7.

Hình dung với Example 1 (`m = n = 3`) trước khi điền:

```text
          j=0   j=1   j=2   j=3
  i=0     ?     ?     ?     INF
  i=1     ?     ?     ?     INF
  i=2     ?     ?     ?     1      <- need[m-1][n] = 1
  i=3     INF   INF   1     INF    <- need[m][n-1] = 1
```

In [18]:
def build_need_table(dungeon):
    """Trả về cả bảng need (m+1) x (n+1) để quan sát."""
    m, n = len(dungeon), len(dungeon[0])
    INF = float("inf")
    need = [[INF] * (n + 1) for _ in range(m + 1)]
    need[m][n - 1] = 1          # hàng rào ngay dưới phòng công chúa
    need[m - 1][n] = 1          # hàng rào ngay phải phòng công chúa
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            nxt = min(need[i + 1][j], need[i][j + 1])     # đi xuống hoặc sang phải
            need[i][j] = max(1, nxt - dungeon[i][j])
    return need


def calculate_dp_table(dungeon: List[List[int]]) -> int:
    return build_need_table(dungeon)[0][0]


print(calculate_dp_table(dungeon1))   # 7
print(calculate_dp_table(dungeon2))   # 1
print(calculate_dp_table(counter2))   # 2

7
1
2


### Trace cách 4 trên Example 1

Thứ tự điền: `(2,2), (2,1), (2,0), (1,2), (1,1), (1,0), (0,2), (0,1), (0,0)`.

| ô | d | need dưới | need phải | nxt = min | need = max(1, nxt − d) |
|---|---:|---:|---:|---:|---:|
| (2,2) | -5 | 1 (rào) | 1 (rào) | 1 | max(1, 1+5) = **6** |
| (2,1) | 30 | INF | 6 | 6 | max(1, 6−30) = **1** |
| (2,0) | 10 | INF | 1 | 1 | max(1, 1−10) = **1** |
| (1,2) | 1 | 6 | INF | 6 | max(1, 6−1) = **5** |
| (1,1) | -10 | 1 | 5 | 1 | max(1, 1+10) = **11** |
| (1,0) | -5 | 1 | 11 | 1 | max(1, 1+5) = **6** |
| (0,2) | 3 | 5 | INF | 5 | max(1, 5−3) = **2** |
| (0,1) | -3 | 11 | 2 | 2 | max(1, 2+3) = **5** |
| (0,0) | -2 | 6 | 5 | 5 | max(1, 5+2) = **7** |

Hai dòng đáng chú ý:

- `(2,1)` có quả cầu +30: theo công thức thô `6 − 30 = −24`, nhưng `max(1, ...)` kéo lên **1** — phải còn sống mới vào được phòng này.
- `(0,0)`: đi xuống cần 6, đi phải cần 5 → chọn **phải**. Đây chính là lý do đường tối ưu bắt đầu bằng `RIGHT`.

Hàm trace dưới in đúng thứ tự và các con số trên.

In [19]:
def trace_dp_table(dungeon):
    m, n = len(dungeon), len(dungeon[0])
    INF = float("inf")
    need = [[INF] * (n + 1) for _ in range(m + 1)]
    need[m][n - 1] = need[m - 1][n] = 1
    print(" ô    |   d | dưới | phải | nxt | need")
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            down, right = need[i + 1][j], need[i][j + 1]
            nxt = min(down, right)
            need[i][j] = max(1, nxt - dungeon[i][j])
            print(f"({i},{j}) | {dungeon[i][j]:>3} | {down:>4} | {right:>4} | {nxt:>3} | {need[i][j]:>4}")
    return need[0][0]


print("Đáp án:", trace_dp_table(dungeon1))

 ô    |   d | dưới | phải | nxt | need
(2,2) |  -5 |    1 |    1 |   1 |    6
(2,1) |  30 |  inf |    6 |   6 |    1
(2,0) |  10 |  inf |    1 |   1 |    1
(1,2) |   1 |    6 |  inf |   6 |    5
(1,1) | -10 |    1 |    5 |   1 |   11
(1,0) |  -5 |    1 |   11 |   1 |    6
(0,2) |   3 |    5 |  inf |   5 |    2
(0,1) |  -3 |   11 |    2 |   2 |    5
(0,0) |  -2 |    6 |    5 |   5 |    7
Đáp án: 7


### Nhìn cả bảng và lần ra đường đi tối ưu

Bảng `need` cuối cùng (bỏ hàng rào) cho Example 1:

```text
  7   5   2
  6  11   5
  1   1   6
```

Muốn biết **đi đường nào**, xuất phát từ `(0,0)` và mỗi bước đi sang ô kế có `need` **nhỏ hơn** (ô đã làm nên giá trị `min` lúc điền bảng). Ở `(0,0)`: dưới 6, phải 5 → `R`. Ở `(0,1)`: dưới 11, phải 2 → `R`. Ở `(0,2)`: chỉ còn đi xuống → `D`, `D`. Ra đúng `RRDD`.

Cuối cùng ta mô phỏng lại bằng `simulate_walk` để chắc chắn: 7 máu sống, 6 máu chết.

In [20]:
def best_path_from_table(dungeon):
    m, n = len(dungeon), len(dungeon[0])
    need = build_need_table(dungeon)
    i, j, moves = 0, 0, ""
    while (i, j) != (m - 1, n - 1):
        if need[i + 1][j] <= need[i][j + 1]:     # ô dưới cần ít máu hơn (hoặc bằng)
            i += 1; moves += "D"
        else:
            j += 1; moves += "R"
    return moves


need1 = build_need_table(dungeon1)
for row in need1[:-1]:
    print(" ".join(f"{v:>3}" for v in row[:-1]))   # bỏ cột / hàng rào

path1 = best_path_from_table(dungeon1)
print("Đường tối ưu:", path1)
print("7 máu sống?", simulate_walk(dungeon1, path1, 7, verbose=False))
print("6 máu sống?", simulate_walk(dungeon1, path1, 6, verbose=False))

  7   5   2
  6  11   5
  1   1   6
Đường tối ưu: RRDD
7 máu sống? True
6 máu sống? False


### Vì sao đúng? (invariant bằng lời)

**Invariant:** ngay sau khi điền ô `(i, j)`, `need[i][j]` **đúng bằng** lượng máu tối thiểu trước cửa phòng `(i, j)` để tới được công chúa.

Lập luận theo thứ tự điền (quy nạp ngược):

1. **Hàng rào đúng nghĩa:** ô ngoài lưới là `INF` — không bao giờ được chọn; hai ô `1` cạnh phòng công chúa biểu diễn yêu cầu "vào phòng cuối xong còn `>= 1` máu".
2. **Khi điền `(i, j)`**, hai ô `(i+1, j)` và `(i, j+1)` đã được điền đúng (vì hàng dưới xong trước, và trong cùng hàng thì cột phải xong trước).
3. **Đủ (không thiếu):** với `h = max(1, nxt − d)` máu, hiệp sĩ sống khi bước vào phòng (`h >= 1`, và `h + d >= nxt >= 1`), rồi đi sang ô kế có `need = nxt` với ít nhất `nxt` máu → theo invariant ở ô kế, chàng tới được công chúa.
4. **Tối thiểu (không dư):** nếu bắt đầu với `h' < max(1, nxt − d)` thì hoặc `h' < 1` (không hợp lệ), hoặc `h' + d < nxt` — tức là ít máu hơn mức tối thiểu của **cả hai** ô kế → đi đường nào cũng chết.

Điểm mấu chốt khác với DP xuôi: `need[i][j]` **không phụ thuộc** ta đã tới `(i, j)` bằng đường nào. Quá khứ chỉ cần "giao nộp" đủ `need[i][j]` máu ở cửa phòng — một con số duy nhất, nên giữ **min** là đủ, không mất thông tin.

### Độ phức tạp cách 4

- **Time:** `O(m × n)` — mỗi ô `O(1)`. Với `200 × 200 = 40 000` ô, chạy tức thì.
- **Space:** `O(m × n)` cho bảng `(m+1) x (n+1)`.

## 9. Cách 5 — DP một chiều: chỉ giữ một hàng

Nhìn lại công thức: khi điền hàng `i`, ta chỉ đọc **hàng `i+1`** (ô dưới) và **chính hàng `i`** (ô phải, vừa điền). Các hàng `i+2, i+3, ...` không bao giờ được đọc lại. Vậy chỉ cần **một mảng `dp` dài `n + 1`**:

- Trước khi cập nhật `dp[j]`, nó còn giữ giá trị của **hàng dưới** → chính là `need dưới`.
- `dp[j+1]` đã được cập nhật ở hàng hiện tại (vì duyệt `j` từ phải sang trái) → chính là `need phải`.

```text
dp[j] = max(1, min(dp[j], dp[j+1]) - dungeon[i][j])
          ^ cũ = dưới   ^ mới = phải
```

Khởi tạo: `dp = [INF] * (n + 1)` và `dp[n-1] = 1` (đóng vai hàng rào ngay dưới phòng công chúa). `dp[n]` luôn là `INF` (cột ngoài lưới).

**Kỳ vọng trên Example 1** (so với bảng ở mục 8, mỗi lần duyệt xong một hàng thì `dp` chính là hàng đó):

| sau hàng | dp |
|---|---|
| khởi tạo | `[inf, inf, 1, inf]` |
| i = 2 | `[1, 1, 6, inf]` |
| i = 1 | `[6, 11, 5, inf]` |
| i = 0 | `[7, 5, 2, inf]` → đáp án `dp[0] = 7` |

In [21]:
def calculate_dp_1d(dungeon: List[List[int]], verbose=False) -> int:
    m, n = len(dungeon), len(dungeon[0])
    INF = float("inf")
    dp = [INF] * (n + 1)
    dp[n - 1] = 1                      # hàng rào dưới phòng công chúa
    if verbose:
        print("khởi tạo :", dp)
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            dp[j] = max(1, min(dp[j], dp[j + 1]) - dungeon[i][j])
        if verbose:
            print(f"sau i = {i}:", dp)
    return dp[0]


print("Đáp án:", calculate_dp_1d(dungeon1, verbose=True))
assert calculate_dp_1d(dungeon2) == 1 and calculate_dp_1d(counter2) == 2

khởi tạo : [inf, inf, 1, inf]
sau i = 2: [1, 1, 6, inf]
sau i = 1: [6, 11, 5, inf]
sau i = 0: [7, 5, 2, inf]
Đáp án: 7


**Độ phức tạp cách 5:**

- **Time:** `O(m × n)` — y như cách 4.
- **Space:** `O(n)` — chỉ một hàng. (Nếu muốn, có thể chọn chiều nhỏ hơn giữa `m` và `n` làm độ dài mảng bằng cách duyệt theo cột, được `O(min(m, n))`.)

Đánh đổi: mất bảng đầy đủ nên **không lần lại được đường đi** như mục 8. Nếu đề chỉ hỏi con số (như bài này) thì 1 chiều là đủ.

## 10. Cách 6 (tuỳ chọn) — Binary search trên đáp án + kiểm tra khả thi

Một góc nhìn hoàn toàn khác, đáng học vì dùng được cho rất nhiều bài "tìm giá trị nhỏ nhất sao cho...":

1. **Đoán** một lượng máu ban đầu `H`.
2. **Kiểm tra** `can_survive(H)`: với `H` máu, có đường nào tới được công chúa không?
3. Nếu `H` đủ thì mọi `H' > H` cũng đủ (nhiều máu hơn không bao giờ thiệt). Tính chất **đơn điệu** (monotonic) này cho phép **binary search** (tìm kiếm nhị phân) tìm `H` nhỏ nhất.

### Kiến thức nền: binary search tìm vị trí `True` đầu tiên

Cho một dãy kiểu `False, False, ..., False, True, True, ..., True`. Muốn tìm chỗ `True` đầu tiên mà không thử từng phần tử: xét điểm giữa `mid`; nếu `True` thì đáp án ở `mid` hoặc bên trái (`hi = mid`); nếu `False` thì đáp án ở bên phải (`lo = mid + 1`). Mỗi bước loại một nửa → `O(log U)` lần kiểm tra, với `U` là độ rộng khoảng tìm.

In [22]:
def first_true(lo, hi, ok):
    """Số nhỏ nhất x trong [lo, hi] có ok(x) == True (giả sử ok đơn điệu và ok(hi) == True)."""
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(mid):
            hi = mid          # mid đủ -> đáp án <= mid
        else:
            lo = mid + 1      # mid chưa đủ -> đáp án > mid
    return lo


# Ví dụ mini: số nhỏ nhất x có x * x >= 50
print(first_true(0, 100, lambda x: x * x >= 50))   # 8 (vì 7*7 = 49 < 50)

8


### Kiểm tra khả thi bằng DP xuôi — lần này DP xuôi lại ĐÚNG!

Với `H` **cố định**, ta đi xuôi từ `(0,0)` và tại mỗi ô lưu `best[i][j]` = **máu nhiều nhất có thể còn lại** sau khi vào phòng `(i, j)` mà vẫn sống (nếu không có cách nào sống tới đó thì lưu `-INF`):

```text
best[i][j] = max(best trên, best trái) + dungeon[i][j]   (chỉ khi max đó > 0, và kết quả > 0)
```

**Tại sao giờ giữ "máu nhiều nhất" lại đúng, còn ở mục 6 thì sai?** Ở mục 6, các đường được so sánh có **máu ban đầu khác nhau** (mỗi đường cần một `H` riêng), nên phải cân hai đại lượng `s` và `low`. Còn ở đây, **mọi đường đều xuất phát cùng `H`**: đường nào còn sống tới `(i, j)` thì "quá khứ" của nó đã được trả đủ, chỉ còn một câu hỏi về tương lai — và với tương lai thì **nhiều máu hơn luôn tốt hơn hoặc bằng**. Một con số, không còn xung đột.

Trên Example 1, ta kỳ vọng `can_survive(H)` là `False` với `H = 1..6` và `True` từ `H = 7`.

In [23]:
def can_survive(dungeon, hp):
    """Với hp máu ban đầu, có đường nào tới công chúa mà không chết không?"""
    m, n = len(dungeon), len(dungeon[0])
    NEG = float("-inf")                    # -inf = không thể sống tới ô này
    best = [[NEG] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            if i == 0 and j == 0:
                before = hp
            else:
                up = best[i - 1][j] if i > 0 else NEG
                left = best[i][j - 1] if j > 0 else NEG
                before = max(up, left)     # máu nhiều nhất khi đứng trước cửa (i, j)
            if before > 0:
                after = before + dungeon[i][j]
                if after > 0:
                    best[i][j] = after
    return best[m - 1][n - 1] > 0


print([(h, can_survive(dungeon1, h)) for h in range(1, 9)])

[(1, False), (2, False), (3, False), (4, False), (5, False), (6, False), (7, True), (8, True)]


**Chọn khoảng tìm `[lo, hi]`:** `lo = 1` (máu tối thiểu hợp lệ). Với `hi`, ta cần một giá trị **chắc chắn đủ**: mọi đường có prefix không nhỏ hơn `−(tổng trị tuyệt đối của các ô âm)`, nên `hi = 1 + tổng(−x với x < 0)` luôn sống. Theo constraints, `hi` tối đa khoảng `1 + 1000 × 40 000`, nên chỉ cần khoảng 26 lần kiểm tra.

In [24]:
def calculate_binary_search(dungeon: List[List[int]]) -> int:
    lo = 1
    hi = 1 + sum(-x for row in dungeon for x in row if x < 0)   # chắc chắn đủ
    return first_true(lo, hi, lambda h: can_survive(dungeon, h))


print(calculate_binary_search(dungeon1))   # 7
print(calculate_binary_search(dungeon2))   # 1
print(calculate_binary_search(counter2))   # 2

7
1
2


**Độ phức tạp cách 6:**

- **Time:** `O(m × n × log U)`, với `U = hi − lo + 1` (độ rộng khoảng tìm). Chậm hơn cách 4 một hệ số `log U` (~26), vẫn qua được constraints nhưng không cần thiết cho bài này.
- **Space:** `O(m × n)` cho bảng `best` (có thể giảm còn `O(n)` như cách 5).

Bài học lớn hơn: khi đề hỏi "giá trị **nhỏ nhất** sao cho **làm được**", và "làm được" có tính đơn điệu, thì **binary search trên đáp án** + **một hàm kiểm tra** thường biến bài tối ưu khó thành bài kiểm tra dễ hơn.

## 11. Code nộp LeetCode

Bản submit dùng cách 4 (DP bảng ngược với hàng rào) vì dễ đọc, dễ kiểm tra. Code submit không `print`, đúng signature `calculateMinimumHP(self, dungeon: List[List[int]]) -> int`.

Ngay dưới là thêm một bản 1 chiều (`SolutionOneRow`) để bạn so sánh — cả hai đều được chấp nhận.

In [25]:
class Solution:
    def calculateMinimumHP(self, dungeon: List[List[int]]) -> int:
        m, n = len(dungeon), len(dungeon[0])
        INF = float("inf")
        need = [[INF] * (n + 1) for _ in range(m + 1)]
        need[m][n - 1] = need[m - 1][n] = 1
        for i in range(m - 1, -1, -1):
            for j in range(n - 1, -1, -1):
                need[i][j] = max(1, min(need[i + 1][j], need[i][j + 1]) - dungeon[i][j])
        return need[0][0]


class SolutionOneRow:
    def calculateMinimumHP(self, dungeon: List[List[int]]) -> int:
        m, n = len(dungeon), len(dungeon[0])
        dp = [float("inf")] * (n + 1)
        dp[n - 1] = 1
        for i in range(m - 1, -1, -1):
            for j in range(n - 1, -1, -1):
                dp[j] = max(1, min(dp[j], dp[j + 1]) - dungeon[i][j])
        return dp[0]


tests = [
    ([[-2, -3, 3], [-5, -10, 1], [10, 30, -5]], 7),   # example 1
    ([[0]], 1),                                       # example 2
    ([[-5]], 6),                                      # 1 ô quỷ: cần 6 để còn 1
    ([[100]], 1),                                     # 1 ô quả cầu: vẫn phải >= 1
    ([[-1000]], 1001),                                # giá trị nhỏ nhất theo constraints
    ([[-1, 5, -10]], 7),                              # 1 hàng: quả cầu trước quỷ giúp ích
    ([[2], [-3], [100], [-1]], 2),                    # 1 cột
    ([[1, 2], [3, 4]], 1),                            # toàn số dương
    ([[0, -1, 3], [0, 0, -3]], 2),                    # phản ví dụ của DP xuôi (B)
    ([[1, -3, 3], [0, -2, 0], [-3, -3, -3]], 3),      # lưới 3x3 khác
]
for solver in (Solution(), SolutionOneRow()):
    for grid_, expected in tests:
        got = solver.calculateMinimumHP(grid_)
        assert got == expected, (type(solver).__name__, grid_, got, expected)
print(f"{len(tests)} test x 2 bản Solution: PASS hết!")

10 test x 2 bản Solution: PASS hết!


### Kiểm thử đối chiếu chéo tất cả các cách

Brute force chậm nhưng **dễ tin là đúng** vì bám sát định nghĩa (thử mọi đường). Ta sinh ngẫu nhiên nhiều lưới nhỏ (cố định `random.seed` để lần nào chạy cũng cùng bộ test) và so mọi cách với nó. Đồng thời đếm xem hai phiên bản DP xuôi sai bao nhiêu lần — để thấy chúng sai **không phải hiếm**.

In [26]:
random.seed(174)
wrong_a = wrong_b = 0
trials = 1500
for _ in range(trials):
    m, n = random.randint(1, 5), random.randint(1, 5)
    g = [[random.randint(-10, 10) for _ in range(n)] for _ in range(m)]
    expected = calculate_bruteforce(g)
    assert calculate_memo(g) == expected, g
    assert calculate_dp_table(g) == expected, g
    assert calculate_dp_1d(g) == expected, g
    assert calculate_binary_search(g) == expected, g
    assert Solution().calculateMinimumHP(g) == expected, g
    assert SolutionOneRow().calculateMinimumHP(g) == expected, g
    assert simulate_walk(g, best_path_from_table(g), expected, verbose=False)
    wrong_a += forward_dp_max_sum(g) != expected
    wrong_b += forward_dp_min_need(g) != expected
print(f"{trials} lưới ngẫu nhiên: 6 cách đúng đều khớp brute force!")
print(f"DP xuôi A sai {wrong_a} lần, DP xuôi B sai {wrong_b} lần.")

1500 lưới ngẫu nhiên: 6 cách đúng đều khớp brute force!
DP xuôi A sai 48 lần, DP xuôi B sai 6 lần.


In [27]:
# Lưới lớn nhất theo constraints: 200 x 200 (bỏ brute force)
big = [[random.randint(-1000, 1000) for _ in range(200)] for _ in range(200)]
for name, f in [("dp_table", calculate_dp_table), ("dp_1d", calculate_dp_1d),
                ("memo", calculate_memo), ("binary_search", calculate_binary_search)]:
    t0 = time.perf_counter()
    ans = f(big)
    print(f"{name:<14}: {ans:>6}  ({(time.perf_counter() - t0) * 1000:7.1f} ms)")

dp_table      :    274  (    5.3 ms)
dp_1d         :    274  (    4.2 ms)
memo          :    274  (   24.4 ms)


binary_search :    274  (  147.0 ms)


## 12. So sánh các cách

Ký hiệu: `m` = số hàng, `n` = số cột, `U` = độ rộng khoảng tìm của binary search.

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Brute force DFS | thử mọi đường, mỗi đường tính `max(1, 1 − min prefix)` | `O(C(m+n−2, m−1) · (m+n))` | `O(m+n)` | chỉ dùng để kiểm chứng |
| 2. DP xuôi giữ 1 cặp | chọn "đường tốt nhất" tới mỗi ô | `O(m·n)` | `O(m·n)` | **SAI** — hai đại lượng `s`, `low` xung đột |
| 3. Đệ quy + memo | `need(i,j)` từ ô về đích | `O(m·n)` | `O(m·n)` + stack `O(m+n)` | dễ viết từ công thức |
| 4. DP bảng ngược | điền `need` từ dưới-phải lên trên-trái | `O(m·n)` | `O(m·n)` | **cách chính**, lần lại được đường đi |
| 5. DP 1 chiều | chỉ giữ 1 hàng `dp` | `O(m·n)` | `O(n)` | gọn bộ nhớ nhất |
| 6. Binary search + check | đoán `H`, DP xuôi kiểm tra khả thi | `O(m·n·log U)` | `O(m·n)` | kỹ thuật tổng quát "tìm min sao cho được" |

## 13. Lỗi người mới hay gặp

### Lỗi 1: Quên `max(1, ...)`

Viết `need = nxt − d` mà không chặn dưới bởi 1. Khi gặp quả cầu lớn, `need` thành số âm, rồi số âm đó "lan" ngược về đầu như thể quả cầu phía sau trả nợ được cho quỷ phía trước. Ví dụ lưới 1 cột `[[-2], [50], [-5]]`: đúng là 3 (vào phòng đầu mất 2 phải còn 1), nhưng bản lỗi trả về **−42**.

### Lỗi 2: Đặt hàng rào bằng 0 thay vì 1

`need[m][n-1] = need[m-1][n] = 0` nghĩa là cho phép vào phòng công chúa xong còn **0 máu** — mà 0 máu là chết. Lưới `[[-5]]`: đúng là 6, bản lỗi trả về 5.

### Lỗi 3: Để mọi ô ngoài lưới là `INF`, quên hai ô hàng rào `1`

Khi đó phòng công chúa có `min(INF, INF) = INF` và cả bảng thành `INF`.

### Lỗi 4: "Máu bằng 0 vẫn sống"

Dùng công thức `max(1, −min(prefix))` (tức coi `hp >= 0` là sống). Example 1 sẽ ra 6 — nhưng mục 3 đã mô phỏng: 6 máu thì tới phòng công chúa còn đúng 0 → chết.

### Lỗi 5: Duyệt bảng sai chiều

Duyệt `i` và `j` **tăng dần** trong khi công thức đọc ô **dưới** và **phải**: lúc điền `(0,0)` thì `(1,0)` và `(0,1)` vẫn là `INF` chưa tính → kết quả sai/`INF`.

### Lỗi 6: Tạo bảng bằng `[[INF] * (n+1)] * (m+1)`

Mọi hàng là cùng một list (mục 8) → ghi vào hàng này là ghi vào mọi hàng, bảng bị trộn lẫn.

### Lỗi 7: Dùng DP xuôi kiểu Minimum Path Sum

Đã phân tích kỹ ở mục 6: Example 1 ra 8 thay vì 7.

Cell dưới chạy từng bản lỗi để bạn thấy tận mắt.

In [28]:
INF = float("inf")


def wrong_no_clamp(dungeon):                 # Lỗi 1
    m, n = len(dungeon), len(dungeon[0])
    need = [[INF] * (n + 1) for _ in range(m + 1)]
    need[m][n - 1] = need[m - 1][n] = 1
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            need[i][j] = min(need[i + 1][j], need[i][j + 1]) - dungeon[i][j]   # thiếu max(1, ...)
    return need[0][0]


def wrong_sentinel_zero(dungeon):            # Lỗi 2
    m, n = len(dungeon), len(dungeon[0])
    need = [[INF] * (n + 1) for _ in range(m + 1)]
    need[m][n - 1] = need[m - 1][n] = 0      # SAI: phải là 1
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            need[i][j] = max(1, min(need[i + 1][j], need[i][j + 1]) - dungeon[i][j])
    return need[0][0]


def wrong_all_inf(dungeon):                  # Lỗi 3
    m, n = len(dungeon), len(dungeon[0])
    need = [[INF] * (n + 1) for _ in range(m + 1)]   # quên hai ô hàng rào = 1
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            need[i][j] = max(1, min(need[i + 1][j], need[i][j + 1]) - dungeon[i][j])
    return need[0][0]


def wrong_zero_is_alive(dungeon):            # Lỗi 4 (trên brute force)
    m, n = len(dungeon), len(dungeon[0])
    best = INF
    for moves in list_paths(m, n):
        prefix = list(accumulate(dungeon[i][j] for i, j in path_cells(moves)))
        best = min(best, max(1, -min(prefix)))      # SAI: phải là 1 - min(prefix)
    return best


def wrong_forward_order(dungeon):            # Lỗi 5
    m, n = len(dungeon), len(dungeon[0])
    need = [[INF] * (n + 1) for _ in range(m + 1)]
    need[m][n - 1] = need[m - 1][n] = 1
    for i in range(m):                       # SAI: duyệt xuôi
        for j in range(n):
            need[i][j] = max(1, min(need[i + 1][j], need[i][j + 1]) - dungeon[i][j])
    return need[0][0]


def wrong_aliased_table(dungeon):            # Lỗi 6
    m, n = len(dungeon), len(dungeon[0])
    need = [[INF] * (n + 1)] * (m + 1)       # SAI: m+1 tham chiếu tới cùng 1 list
    need[m][n - 1] = need[m - 1][n] = 1
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            need[i][j] = max(1, min(need[i + 1][j], need[i][j + 1]) - dungeon[i][j])
    return need[0][0]


col = [[-2], [50], [-5]]
print("Lỗi 1:", col, "đúng =", calculate_dp_table(col), "| sai =", wrong_no_clamp(col))
print("Lỗi 2: [[-5]] đúng =", calculate_dp_table([[-5]]), "| sai =", wrong_sentinel_zero([[-5]]))
print("Lỗi 3: example 1 đúng =", calculate_dp_table(dungeon1), "| sai =", wrong_all_inf(dungeon1))
print("Lỗi 4: example 1 đúng =", calculate_dp_table(dungeon1), "| sai =", wrong_zero_is_alive(dungeon1))
print("Lỗi 5: example 1 đúng =", calculate_dp_table(dungeon1), "| sai =", wrong_forward_order(dungeon1))
print("Lỗi 6: example 1 đúng =", calculate_dp_table(dungeon1), "| sai =", wrong_aliased_table(dungeon1))
print("Lỗi 7: example 1 đúng =", calculate_dp_table(dungeon1), "| sai =", forward_dp_max_sum(dungeon1))

Lỗi 1: [[-2], [50], [-5]] đúng = 3 | sai = -42
Lỗi 2: [[-5]] đúng = 6 | sai = 5
Lỗi 3: example 1 đúng = 7 | sai = inf
Lỗi 4: example 1 đúng = 7 | sai = 6
Lỗi 5: example 1 đúng = 7 | sai = inf
Lỗi 6: example 1 đúng = 7 | sai = 6
Lỗi 7: example 1 đúng = 7 | sai = 8


## 14. Mẹo nhận dạng pattern

- Đề hỏi **"giá trị ban đầu tối thiểu"** (máu, tiền, nhiên liệu...) sao cho **ở mọi thời điểm** một đại lượng không được tụt dưới ngưỡng → nghĩ tới **DP ngược từ đích**: "đứng ở đây cần bao nhiêu để đi nốt?".
- Thấy dạng **`max(ngưỡng, nhu cầu phía sau − giá trị hiện tại)`**: đó là dấu hiệu "nhu cầu" lan ngược và bị **chặn dưới**.
- Nếu đi xuôi mà mỗi trạng thái cần **hai con số mâu thuẫn** để so sánh (một cái tốt cho quá khứ, một cái tốt cho tương lai) → thử **đổi chiều DP** để chỉ còn một con số.
- Câu hỏi "**nhỏ nhất sao cho làm được**" với tính đơn điệu → **binary search trên đáp án** + hàm kiểm tra (cách 6).
- Lưới chỉ đi **phải / xuống** → luôn có thứ tự điền bảng không vòng lặp (xuôi hoặc ngược), DP `O(m·n)`; thường giảm được còn **1 hàng**.

## 15. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| LeetCode 1413 — Minimum Value to Get Positive Step by Step Sum | Phiên bản 1 chiều (1 đường cố định): `max(1, 1 − min prefix)` — chính là mục 3 |
| LeetCode 64 — Minimum Path Sum | Cùng lưới phải/xuống, nhưng DP **xuôi** đúng vì chỉ có 1 đại lượng (tổng) |
| LeetCode 62 / 63 — Unique Paths (I, II) | Đếm số đường trên lưới — liên quan `C(m+n−2, m−1)` ở mục 4 |
| LeetCode 120 — Triangle | DP từ dưới lên (bottom-up), 1 chiều |
| LeetCode 741 — Cherry Pickup | DP trên lưới khó hơn: hai người đi cùng lúc |
| LeetCode 871 — Minimum Number of Refueling Stops | Cũng là bài "nhiên liệu không được cạn giữa đường" |

## 16. Bài tập vận dụng

Làm theo thứ tự, từ dễ tới khó. Gợi ý ở mục 17, đáp án ở **cuối notebook**.

### Bài 1 — Tính tay bảng `need`

Cho

```text
dungeon = [[ 1, -3,  3],
           [ 0, -2,  0],
           [-3, -3, -3]]
```

Lập bảng `need` 3 x 3 (điền theo thứ tự từ `(2,2)` ngược về `(0,0)` như trace mục 8), cho biết đáp án và đường đi tối ưu.

### Bài 2 — Đoán output (không chạy code)

Mỗi lưới sau cần tối thiểu bao nhiêu máu?

- (a) `[[-5]]`
- (b) `[[5]]`
- (c) `[[-1, 5, -10]]`
- (d) `[[2], [-3], [100], [-1]]`
- (e) `[[0, 0], [0, 0]]`

Ở câu (c), nếu đổi thứ tự thành `[[-1, -10, 5]]` thì đáp án thay đổi thế nào? Vì sao?

### Bài 3 — Điền TODO: DP bảng ngược

Hoàn thiện hàm theo 3 bước: tạo bảng có hàng rào, duyệt ngược, áp công thức.

In [29]:
def exercise_dp_table(dungeon: List[List[int]]) -> int:
    m, n = len(dungeon), len(dungeon[0])
    INF = float("inf")
    # TODO 1: tạo bảng need kích thước (m+1) x (n+1) toàn INF (nhớ dùng list comprehension!)
    # TODO 2: đặt hai ô hàng rào cạnh phòng công chúa bằng 1
    # TODO 3: duyệt i từ m-1 về 0, j từ n-1 về 0, áp need = max(1, min(dưới, phải) - dungeon[i][j])
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_dp_table([[-2,-3,3],[-5,-10,1],[10,30,-5]]) == 7
# assert exercise_dp_table([[0]]) == 1
# assert exercise_dp_table([[1,-3,3],[0,-2,0],[-3,-3,-3]]) == 3

### Bài 4 — Tự viết lại DP một chiều theo **cột**

Ở mục 9 ta giữ một **hàng** (độ dài `n + 1`). Hãy viết phiên bản giữ một **cột** (độ dài `m + 1`): duyệt `j` từ `n-1` về 0 ở vòng ngoài, `i` từ `m-1` về 0 ở vòng trong. Phiên bản này có bộ nhớ `O(m)` — hữu ích khi lưới rất "rộng" (`n` lớn hơn nhiều so với `m`).

Câu hỏi tư duy: lúc này `dp[i]` **cũ** đóng vai ô nào (dưới hay phải), và `dp[i+1]` **mới** đóng vai ô nào? Khởi tạo hàng rào ra sao?

In [30]:
def exercise_dp_1d_column(dungeon: List[List[int]]) -> int:
    m, n = len(dungeon), len(dungeon[0])
    INF = float("inf")
    # TODO: dp dài m + 1, đặt hàng rào, hai vòng lặp (j ngoài, i trong), trả về dp[0]
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_dp_1d_column([[-2,-3,3],[-5,-10,1],[10,30,-5]]) == 7
# assert exercise_dp_1d_column([[-1, 5, -10]]) == 7
# assert exercise_dp_1d_column([[2], [-3], [100], [-1]]) == 2

### Bài 5 — Biến thể: trả về cả đường đi

Viết hàm trả về cặp `(hp, moves)`: `hp` là máu tối thiểu, `moves` là một chuỗi `'R'/'D'` của **một** đường tối ưu (nếu có nhiều đường tối ưu, trả về đường nào cũng được).

Yêu cầu kiểm chứng: đi theo `moves` với `hp` máu thì **sống**, với `hp − 1` máu thì **chết** (dùng `simulate_walk(..., verbose=False)`).

```python
exercise_min_hp_with_path([[-2,-3,3],[-5,-10,1],[10,30,-5]]) -> (7, 'RRDD')
```

In [31]:
def exercise_min_hp_with_path(dungeon: List[List[int]]):
    # TODO 1: dựng bảng need như cách 4
    # TODO 2: từ (0,0), mỗi bước đi sang ô kế (dưới/phải) có need nhỏ hơn, ghi lại 'D'/'R'
    # TODO 3: return need[0][0], moves
    pass

# Tự làm xong rồi mới mở test:
# hp, mv = exercise_min_hp_with_path([[-2,-3,3],[-5,-10,1],[10,30,-5]])
# assert hp == 7 and simulate_walk([[-2,-3,3],[-5,-10,1],[10,30,-5]], mv, hp, verbose=False)
# assert not simulate_walk([[-2,-3,3],[-5,-10,1],[10,30,-5]], mv, hp - 1, verbose=False)

### Bài 6 — Mini challenge: LeetCode 64 — Minimum Path Sum

Cho lưới `grid` các số **không âm**, đi từ trên-trái tới dưới-phải (chỉ phải / xuống), tìm **tổng nhỏ nhất** của các ô trên đường đi.

Yêu cầu: DP **xuôi** từ `(0,0)`, bộ nhớ `O(n)` (một hàng).

Câu hỏi tư duy: vì sao ở bài 64 DP xuôi giữ "tổng nhỏ nhất tới mỗi ô" lại **đúng**, trong khi ở bài 174 DP xuôi lại sai? (Gợi ý: đếm số đại lượng cần nhớ cho mỗi đường.)

```python
min_path_sum([[1,3,1],[1,5,1],[4,2,1]]) -> 7
min_path_sum([[1,2,3],[4,5,6]]) -> 12
```

In [32]:
def exercise_min_path_sum(grid: List[List[int]]) -> int:
    # TODO: dp[j] = tổng nhỏ nhất tới ô (i, j); dp[j] = grid[i][j] + min(dp[j] (trên), dp[j-1] (trái))
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_min_path_sum([[1,3,1],[1,5,1],[4,2,1]]) == 7
# assert exercise_min_path_sum([[1,2,3],[4,5,6]]) == 12

## 17. Gợi ý theo mức

**Bài 1**
- Mức 1: bắt đầu từ phòng công chúa: `need = max(1, 1 − d)`.
- Mức 2: hàng cuối chỉ có "phải", cột cuối chỉ có "dưới" (phía còn lại là `INF`).
- Mức 3: `need[2][2] = 4`, `need[2][1] = 7`, `need[1][2] = 4` — tự điền tiếp 6 ô còn lại.

**Bài 2**
- Mức 1: một đường duy nhất → dùng công thức `max(1, 1 − min prefix)` ở mục 3.
- Mức 2: (c) prefix là `-1, 4, -6`.
- Mức 3: khi đổi thứ tự `[-1, -10, 5]`, prefix là `-1, -11, -6` — đáy sâu hơn.

**Bài 3**
- Mức 1: chép lại khung `build_need_table` ở mục 8, đừng nhìn — viết theo trí nhớ.
- Mức 2: `need = [[INF] * (n + 1) for _ in range(m + 1)]`.
- Mức 3: `need[m][n - 1] = need[m - 1][n] = 1`; vòng lặp `for i in range(m - 1, -1, -1)`.

**Bài 4**
- Mức 1: giờ vòng ngoài là cột → mảng lưu một **cột** của bảng `need`.
- Mức 2: `dp[i]` cũ là giá trị của cột `j+1` cùng hàng → ô **phải**; `dp[i+1]` vừa cập nhật ở cột `j` → ô **dưới**.
- Mức 3: `dp = [INF] * (m + 1)`; `dp[m - 1] = 1` (hàng rào bên phải phòng công chúa); `dp[i] = max(1, min(dp[i], dp[i + 1]) - dungeon[i][j])`.

**Bài 5**
- Mức 1: dùng lại `build_need_table`.
- Mức 2: vòng `while (i, j) != (m - 1, n - 1)`; nhờ hàng rào `INF`, so sánh `need[i+1][j]` với `need[i][j+1]` tự động không bao giờ đi ra ngoài.
- Mức 3: xem lại `best_path_from_table` ở mục 8.

**Bài 6**
- Mức 1: ô hàng đầu chỉ đến được từ trái; ô cột đầu chỉ đến được từ trên.
- Mức 2: khởi tạo `dp = [INF] * n`, `dp[0] = 0` trước hàng đầu thì công thức chung xử lý được cả hàng đầu.
- Mức 3: với `j = 0` chỉ lấy `dp[0]` (trên); với `j > 0` lấy `min(dp[j], dp[j - 1])`.

## 18. Đáp án tham khảo

Hãy tự làm trước rồi mới xem phần này.

**Bài 1** — thứ tự điền và kết quả:

| ô | d | dưới | phải | need |
|---|---:|---:|---:|---:|
| (2,2) | -3 | 1 | 1 | 4 |
| (2,1) | -3 | INF | 4 | 7 |
| (2,0) | -3 | INF | 7 | 10 |
| (1,2) | 0 | 4 | INF | 4 |
| (1,1) | -2 | 7 | 4 | 6 |
| (1,0) | 0 | 10 | 6 | 6 |
| (0,2) | 3 | 4 | INF | 1 |
| (0,1) | -3 | 6 | 1 | 4 |
| (0,0) | 1 | 6 | 4 | **3** |

Bảng:

```text
  3   4   1
  6   6   4
 10   7   4
```

Đáp án **3**, đường `RRDD` (giá trị `1, -3, 3, 0, -3`, prefix `1, -2, 1, 1, -2`, đáy -2 → `max(1, 3) = 3`). Để ý `(0,2)` có `need = 1` nhờ quả cầu +3 bù cho cú -3 ở cuối.

**Bài 2**

| lưới | prefix | đáp án |
|---|---|---:|
| (a) `[[-5]]` | -5 | **6** |
| (b) `[[5]]` | 5 | **1** |
| (c) `[[-1, 5, -10]]` | -1, 4, -6 | **7** |
| (d) `[[2],[-3],[100],[-1]]` | 2, -1, 99, 98 | **2** |
| (e) `[[0,0],[0,0]]` | 0, 0, 0 | **1** |

Đổi (c) thành `[[-1, -10, 5]]`: prefix `-1, -11, -6`, đáp án **12**. Tổng cả đường như nhau (-6), nhưng quả cầu đến **sau** con quỷ -10 thì không cứu được — thứ tự quan trọng, nên bài này không thể chỉ nhìn tổng.

**Bài 6 — câu hỏi tư duy:** ở bài 64, chi phí tương lai **cộng thẳng** vào tổng hiện tại, và một đường tới `(i, j)` chỉ được mô tả bởi **một** con số (tổng); đường có tổng nhỏ hơn luôn tốt hơn bất kể phía sau. Ở bài 174, một đường cần **hai** con số (`s` và `low`) và không có thứ tự "tốt hơn" chung cho cả hai, nên DP xuôi giữ 1 giá trị bị mất thông tin.

Cell dưới kiểm chứng bài 1, bài 2 bằng máy, sau đó là đáp án code bài 3–6.

In [33]:
# Kiểm chứng Bài 1 và Bài 2
ex1 = [[1, -3, 3], [0, -2, 0], [-3, -3, -3]]
_ = trace_dp_table(ex1)
print("Đường tối ưu bài 1:", best_path_from_table(ex1))
for g, expected in [([[-5]], 6), ([[5]], 1), ([[-1, 5, -10]], 7),
                    ([[2], [-3], [100], [-1]], 2), ([[0, 0], [0, 0]], 1), ([[-1, -10, 5]], 12)]:
    assert calculate_dp_table(g) == expected == calculate_bruteforce(g), g
print("Bài 2: tất cả đáp án khớp!")

 ô    |   d | dưới | phải | nxt | need
(2,2) |  -3 |    1 |    1 |   1 |    4
(2,1) |  -3 |  inf |    4 |   4 |    7
(2,0) |  -3 |  inf |    7 |   7 |   10
(1,2) |   0 |    4 |  inf |   4 |    4
(1,1) |  -2 |    7 |    4 |   4 |    6
(1,0) |   0 |   10 |    6 |   6 |    6
(0,2) |   3 |    4 |  inf |   4 |    1
(0,1) |  -3 |    6 |    1 |   1 |    4
(0,0) |   1 |    6 |    4 |   4 |    3
Đường tối ưu bài 1: RRDD
Bài 2: tất cả đáp án khớp!


In [34]:
def answer_dp_table(dungeon: List[List[int]]) -> int:
    m, n = len(dungeon), len(dungeon[0])
    INF = float("inf")
    need = [[INF] * (n + 1) for _ in range(m + 1)]
    need[m][n - 1] = need[m - 1][n] = 1
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            need[i][j] = max(1, min(need[i + 1][j], need[i][j + 1]) - dungeon[i][j])
    return need[0][0]


def answer_dp_1d_column(dungeon: List[List[int]]) -> int:
    m, n = len(dungeon), len(dungeon[0])
    INF = float("inf")
    dp = [INF] * (m + 1)
    dp[m - 1] = 1                       # hàng rào bên phải phòng công chúa
    for j in range(n - 1, -1, -1):
        for i in range(m - 1, -1, -1):
            # dp[i] cũ = ô phải (cột j+1), dp[i+1] mới = ô dưới (cột j)
            dp[i] = max(1, min(dp[i], dp[i + 1]) - dungeon[i][j])
    return dp[0]


def answer_min_hp_with_path(dungeon: List[List[int]]):
    m, n = len(dungeon), len(dungeon[0])
    INF = float("inf")
    need = [[INF] * (n + 1) for _ in range(m + 1)]
    need[m][n - 1] = need[m - 1][n] = 1
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            need[i][j] = max(1, min(need[i + 1][j], need[i][j + 1]) - dungeon[i][j])
    i, j, moves = 0, 0, ""
    while (i, j) != (m - 1, n - 1):
        if need[i + 1][j] <= need[i][j + 1]:
            i += 1; moves += "D"
        else:
            j += 1; moves += "R"
    return need[0][0], moves


def answer_min_path_sum(grid: List[List[int]]) -> int:
    n = len(grid[0])
    INF = float("inf")
    dp = [INF] * n
    dp[0] = 0                           # "trước hàng đầu": chỉ ô (0,0) đến được
    for row in grid:
        for j in range(n):
            from_up = dp[j]
            from_left = dp[j - 1] if j > 0 else INF
            dp[j] = row[j] + min(from_up, from_left)
    return dp[-1]


assert answer_dp_table(dungeon1) == 7 and answer_dp_table([[0]]) == 1 and answer_dp_table(ex1) == 3
assert answer_dp_1d_column(dungeon1) == 7
assert answer_dp_1d_column([[-1, 5, -10]]) == 7
assert answer_dp_1d_column([[2], [-3], [100], [-1]]) == 2
hp, mv = answer_min_hp_with_path(dungeon1)
assert (hp, mv) == (7, "RRDD")
assert answer_min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]) == 7
assert answer_min_path_sum([[1, 2, 3], [4, 5, 6]]) == 12

# Đối chiếu ngẫu nhiên các đáp án với brute force
random.seed(2026)
for _ in range(500):
    m, n = random.randint(1, 5), random.randint(1, 5)
    g = [[random.randint(-8, 8) for _ in range(n)] for _ in range(m)]
    expected = calculate_bruteforce(g)
    assert answer_dp_table(g) == expected, g
    assert answer_dp_1d_column(g) == expected, g
    hp, mv = answer_min_hp_with_path(g)
    assert hp == expected and simulate_walk(g, mv, hp, verbose=False), g
    assert hp == 1 or not simulate_walk(g, mv, hp - 1, verbose=False), g
    pos = [[abs(x) for x in row] for row in g]
    brute_sum = min(sum(pos[i][j] for i, j in path_cells(p)) for p in list_paths(m, n))
    assert answer_min_path_sum(pos) == brute_sum, pos
print("Đáp án tham khảo chạy PASS toàn bộ test!")

Đáp án tham khảo chạy PASS toàn bộ test!


## 19. Template ngắn nên nhớ

```python
class Solution:
    def calculateMinimumHP(self, dungeon: List[List[int]]) -> int:
        m, n = len(dungeon), len(dungeon[0])
        INF = float("inf")
        need = [[INF] * (n + 1) for _ in range(m + 1)]   # thêm hàng rào
        need[m][n - 1] = need[m - 1][n] = 1             # vào phòng cuối xong còn >= 1
        for i in range(m - 1, -1, -1):                  # đi NGƯỢC từ đích
            for j in range(n - 1, -1, -1):
                need[i][j] = max(1, min(need[i + 1][j], need[i][j + 1]) - dungeon[i][j])
        return need[0][0]
```

Câu thần chú: **"Đứng trước cửa phòng, cần bao nhiêu máu để đi nốt? Lấy ô kế rẻ hơn, trừ giá trị phòng, nhưng không bao giờ dưới 1."**

## 20. Checklist trước khi submit

- Mình có duyệt **ngược** (từ `m-1, n-1` về `0, 0`) chứ không phải xuôi?
- Có `max(1, ...)` ở **mọi** ô (kể cả phòng công chúa) chưa?
- Hai ô hàng rào cạnh phòng công chúa là **1** (không phải 0, không phải `INF`)?
- Bảng tạo bằng list comprehension, không phải `[[...]] * (m + 1)`?
- Đã thử `[[0]] → 1`, `[[-5]] → 6`, `[[100]] → 1`, lưới 1 hàng, lưới 1 cột và Example 1 `→ 7`?
- Nếu nhỡ nghĩ tới DP xuôi: nhớ phản ví dụ Example 1 (DP xuôi "max máu dư" ra 8).